# Installations:

## Install Packages
*   **Unsloth:** An innovative software library designed to significantly enhance the fine-tuning of large language models (LLMs).
*   **xFormers:** xFormers is an advanced, open-source library developed by the FAIR team at Meta AI, designed to accelerate transformer model research and applications across various fields.
*   **TRL:** A comprehensive library developed for training transformer language models using reinforcement learning techniques. It's built on top of the Hugging Face Transformers library and supports a variety of tasks including Supervised Fine-Tuning (SFT), Reward Modeling (RM), and Proximal Policy Optimization (PPO), among others.
*   **PEFT:** PEFT offers parameter-efficient methods for finetuning large pretrained models. It trains a smaller number of prompt parameters or use a reparametrization method like low-rank adaptation (LoRA) to reduce the number of trainable parameters.
*   **Accelerate:** Accelerate is a library that enables the same PyTorch code to be run across any distributed configuration by adding just four lines of code! In short, training and inference at scale made simple, efficient and adaptable.
*   **Bitsandbytes:** Bitsandbytes enables accessible large language models via k-bit quantization for PyTorch.



In [ ]:
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install --upgrade peft
!pip install --upgrade accelerate #trl

In [ ]:
!pip install --upgrade \
    transformers \
    bitsandbytes \
    huggingface_hub \
    safetensors \
    tokenizers \
    sentencepiece \
    protobuf \
    scipy

In [ ]:
!pip install datasets==4.3.0
!pip install trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
print("Unsloth loaded successfully!")
import torch
print(torch.__version__)
print(torch.cuda.is_available())


In [ ]:
from google.colab import userdata
hf_key=userdata.get('hf_key')
from huggingface_hub import login

login(hf_key)


In [ ]:
!pip install ipython-autotime

In [ ]:
load_ext autotime

In [ ]:
from IPython.display import Markdown, HTML
import numpy as np

In [ ]:
#connect to drive:
from google.colab import drive
drive.mount('/content/drive')

##Install Packages:

#Access through API


# Finetune LLM

## Load Model and Tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,

)

In [ ]:
import psutil

# Get the RAM information
ram_gb = psutil.virtual_memory().total / 1e9

print("Total RAM available: {:.2f} GB".format(ram_gb))


## Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, #16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32, #16,
    lora_dropout = 0.1, #0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Print the model to see the applied LoRA adapters.

In [ ]:
print(model)

# Prepare Data:

In [ ]:
import pandas as pd

train= pd.read_csv("/content/train2.csv")
dev= pd.read_csv("/content/dev2.csv")

instruction_col = []
# text = "Act as a Natural Language processing expert in Arabic Language, your task is to to extract the aspect term/terms that are being described in the following sentence. Mostly, the identified aspect can be a noun or a noun phrase, your answer should contain the extracted aspect terms in Arabic."
text = """Act as a Natural Language Processing expert in Arabic language.
Your task is to extract aspect term(s) that are explicitly mentioned in the given sentence.
- Only return aspect terms that appear verbatim in the sentence.
- Do not infer or invent related words.
- If no aspect is found, return "none"."""
for _, row in train.iterrows():
  instruction_col.append(text)

train['instruction'] = text
dev['instruction'] = text


instruction_col = []


In [ ]:
type(train.iloc[0]['aspect'])

In [ ]:
#Change train['aspect'] from float "NaN" to the string "none":
train['aspect'] = train['aspect'].apply(lambda x: 'none' if pd.isna(x) else x)
train

In [ ]:
prompt = """Below is an instruction that describes a task, paired with an input Sentence that provides a text. Write a response that appropriately completes the request.
###Instruction:
{}
###Sentence:
{}
###Aspect:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_row(row):
    return prompt.format(row["instruction"], row["text"], row["aspect"]) + EOS_TOKEN

train["text"] = train.apply(formatting_prompts_row, axis=1)
dev["text"]  = dev.apply(formatting_prompts_row, axis=1)


In [ ]:
from datasets import Dataset

# Convert your pandas DataFrame into a Hugging Face Dataset
train_dataset = Dataset.from_pandas(train)
eval_dataset  = Dataset.from_pandas(dev)

print(train_dataset)
print(eval_dataset)


# Train!

In [ ]:
# Calculate required max_steps: steps_per_epoch=dataset size / batch size

get the number of rows in train_dataset:
dataset_size= len(train_dataset)
print("Dataset size: " ,dataset_size)
steps_per_epoch = dataset_size // 8
max_steps = steps_per_epoch * 6


In [ ]:
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 1,
    warmup_steps = 2,

    max_steps = steps_per_epoch * 3, #  (Here we specified that 6 epochs are enough.)
    # max_steps = 675, # How to calculate: steps_per_epoch=dataset size / batch size,    max_steps = steps_per_epoch * 5 (Here we specified that 5 epochs are enough.)
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 50,   # report logs every 50 steps
    eval_strategy = "steps",   # <--- add this
    eval_steps = 200,                 # evaluate every 200 steps
    save_strategy = "steps",          # must match
    save_steps = 200,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",  # <--- or any metric from compute_metrics
    greater_is_better = False,            # because lower loss is better
    optim = "paged_adamw_32bit",
    seed = 3407,
    output_dir = "outputs",
    learning_rate = 1e-4,
    weight_decay = 0.0,
    lr_scheduler_type = "linear"
)


In [ ]:
model.train()
for n, p in model.named_parameters():
    if p.requires_grad:
        print("Trainable:", n)


In [ ]:
from trl import SFTTrainer
import os
os.environ["WANDB_API_KEY"] = "......."


# 3) Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset= train_dataset,
    eval_dataset=eval_dataset,

    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = training_arguments,

    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)],
    #patience=2 → stop after 2 evals (≈400 steps) with no improvement
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
import psutil

# Get the RAM information
ram_gb = psutil.virtual_memory().total / 1e9

print("Total RAM available: {:.2f} GB".format(ram_gb))


##Save Model:

##Upload the Model to HF

In [ ]:
from google.colab import userdata
# # openai_key=userdata.get('openAi_key')
# myGPT = userdata.get('MyGPT')
hf_key=userdata.get('hf_key')

# from huggingface_hub import notebook_login
# notebook_login()
from huggingface_hub import login

# paste your token string (found in your Hugging Face account settings)
login(hf_key)

In [ ]:
model.push_to_hub("LujainAbdulrahman/finetuned_m-absa_3epochs_noEarlyStopping ")#,use_auth_token= hf_key)

tokenizer.push_to_hub("LujainAbdulrahman/finetuned_m-absa_3epochs_noEarlyStopping ")#,use_auth_token= hf_key)